# 06_作業_辨識自己的物體.py
**電腦視覺與深度學習 › Day18 CNN + Teachable Machine + MediaPipe**

這是 Day18 的作業骨架:接上你自己訓練的 Teachable Machine 模型,再加上自己的創意,例如駝背太久就提醒你。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/3)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY9"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '06_作業_辨識自己的物體.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 需要你自己的模型
這支要用你在 Teachable Machine 匯出的 `keras_model.h5`、`labels.txt`。執行下一格後按「選擇檔案」上傳。

In [ ]:
#@title 上傳模型檔
from google.colab import files
files.upload()

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 四個 TODO
開頭列出四個步驟:想一個有意義的分類任務、匯出模型、在自訂邏輯區加上你的應用,最後錄一段 30 秒的展示影片。

In [ ]:
# 06 — 作業：接你自己訓練的 Teachable Machine 模型，做一個有實際用途的應用

### 匯入與路徑
一樣要在 import tensorflow 之前設定 TF_USE_LEGACY_KERAS,再組出模型和標籤的路徑。

In [ ]:
#
# TODO 1) 到 Teachable Machine 訓練你自己的模型（見 02 教學）
#         想一個有意義的分類任務，例如：
#           - 口罩：戴 / 沒戴 / 戴錯位置
#           - 姿勢：坐姿正 / 駝背 / 站立
#           - 手勢：剪刀 / 石頭 / 布（拳皇對戰？）
#           - 情緒：微笑 / 皺眉 / 中性
#           - 桌面整齊度：乾淨 / 亂
#
# TODO 2) 匯出 → 把 keras_model.h5 + labels.txt 放到本資料夾
#
# TODO 3) 在下方 [自訂邏輯] 區段加上你的創意應用，例如：
#         - 分類結果變化時發聲（winsound / playsound）
#         - 記錄每次分類到 SQLite（結合 DAY6 web_basics）
#         - 觸發 GPIO / 開燈 / 顯示到別的視窗
#         - 分類是「駝背」超過 5 秒 → 跳警告視窗
#
# TODO 4) 交作業時：附上你的 model + 一段 30 秒的 demo 影片
#
# 需要：pip install tensorflow opencv-python numpy

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # 必須在 import tensorflow 之前

import time
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont

BASE = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH  = os.path.join(BASE, "keras_model.h5")

### 畫中文的小幫手
找中文字型,找不到就提早報錯。畫中文函式和 03 那支一樣,先轉 RGB 給 Pillow 畫,再轉回來。

In [ ]:
LABELS_PATH = os.path.join(BASE, "labels.txt")

# ====== 中文字型（OpenCV putText 不支援中文，改用 PIL）======
FONT_CANDIDATES = [
    "C:/Windows/Fonts/msjh.ttc",   # Microsoft JhengHei
    "C:/Windows/Fonts/msyh.ttc",   # Microsoft YaHei
    "C:/Windows/Fonts/simhei.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
if FONT_PATH is None:
    raise RuntimeError("找不到中文字型，請安裝微軟正黑體或改字型路徑")

### 載入模型與開鏡頭
讀標籤、讀模型、打開鏡頭。

In [ ]:
def 畫中文(img_bgr, text, xy, size, color_bgr):
    """在 OpenCV BGR 影像上畫中文，color 用 BGR"""
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(FONT_PATH, size)
    # PIL 用 RGB，把 BGR 換過來
    draw.text(xy, text, font=font, fill=(color_bgr[2], color_bgr[1], color_bgr[0]))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# ====== 讀模型 + labels ======
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    labels = [line.strip().split(" ", 1)[1] for line in f if line.strip()]

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

cap = cv2.VideoCapture(0)

### 記住上一次的狀態
要知道「同一個類別持續多久」,就得記住上一次的類別和時間。這三個變數就是程式的記憶。

In [ ]:
if not cap.isOpened():
    raise RuntimeError("無法開啟 webcam")

# ====== 狀態變數：追蹤上一次的類別和累積時間 ======
上次類別   = None
上次時間   = time.time()
持續秒數   = 0.0

### 前處理與推論
和 03 完全相同的前處理,再取出機率最大的類別和信心分數。

### 計算持續秒數
算出這一格和上一格差幾秒。如果類別沒變,就把秒數累加上去;換了類別就歸零,重新計時。

### 你的創意放這裡
這裡是留給你發揮的地方。範例是:駝背超過 3 秒就印警告、看到剪刀就嗶一聲。把註解打開、類別名稱改成你自己的就能用。

### 畫面顯示與收拾
用綠色大字寫出類別和信心,黃色小字寫出持續秒數,再顯示畫面,按 q 離開。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 預處理
    img = cv2.resize(frame, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    img = (img / 127.5) - 1
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img, verbose=0)[0]
    top_idx = int(np.argmax(preds))
    目前類別 = labels[top_idx]
    信心    = float(preds[top_idx])

    # ====== 累積目前類別持續多久 ======
    now = time.time()
    dt = now - 上次時間
    上次時間 = now
    if 目前類別 == 上次類別:
        持續秒數 += dt
    else:
        持續秒數 = 0.0
        上次類別 = 目前類別

    # ====== [自訂邏輯] 在這裡加你自己的動作！======
    # 範例：某個類別持續 3 秒以上就印警告
    # if 目前類別 == "駝背" and 持續秒數 > 3.0:
    #     print(">>> 警告：駝背太久了！")
    #
    # 範例：偵測到特定類別發聲（Windows）
    # if 目前類別 == "剪刀" and 持續秒數 > 1.0:
    #     import winsound
    #     winsound.Beep(1000, 100)

    # ====== 畫面顯示（用 PIL 畫中文）======
    frame = 畫中文(frame, f"{目前類別} ({信心*100:.1f}%)",
                   (10, 10), 32, (0, 255, 0))
    frame = 畫中文(frame, f"持續: {持續秒數:.1f}s",
                   (10, 55), 24, (0, 255, 255))

    cv2.imshow("我的分類器應用", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 小結
作業的重點是把分類結果變成「動作」。完成後,下一支手寫辨識會把 CNN 接上 tkinter 畫布。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/3) 看逐句拆解、做練習。